In [ ]:
import pandas as pd

# 1. 학생 성적 데이터 (students.csv)
data1 = {
    'student_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'name': ['Kim', 'Lee', 'Park', 'Choi', 'Jung', 'Kang', 'Jo', 'Yoon'],
    'gender_code': ['M', 'F', 'M', 'M', 'F', 'F', 'M', 'F'],
    'game_hours': [5, 2, 8, 1, 3, 7, 4, 2],
    'math_score': [45, 88, 30, 95, 75, 40, 60, 90],
    'english_score': [55, 90, 40, 85, 80, 45, 65, 95]
}
pd.DataFrame(data1).to_csv('students.csv', index=False)

# 2. 학생 방과 후 활동 데이터 (after_school.csv) - 병합(merge)용
data2 = {
    'student_id': [1, 2, 3, 4, 5, 6, 7, 8],
    'activity': ['Soccer', 'Art', 'Gaming', 'Reading', 'Art', 'Gaming', 'Soccer', 'Reading']
}
pd.DataFrame(data2).to_csv('after_school.csv', index=False)

print("실습용 CSV 파일이 생성되었습니다!")

**Pandas 1: 데이터 불러오기 및 기본 조작**

In [ ]:
# 1. CSV 파일 읽기 및 특정 컬럼 접근
df = pd.read_csv('students.csv')
df.head()

# gender만 출력

# student_id와 name만 출력

In [ ]:
# 2. 새로운 컬럼 생성 (총점 계산)
df["total_score"] = df["math_score"] + df["english_score"]

df.head()

In [ ]:
# 3. map을 활용한 딕셔너리 매핑 (M, F -> Male, Female)
gender_dict = {'M': 'Male', 'F': 'Female'}
df['gender'] = df['gender_code'].map(gender_dict)

df.head()

**Pandas 2: 필터링, loc, apply**

In [ ]:
# 4. 필터링: 수학 점수가 80점 이상인 학생
high_math = df[df["math_score"] >= 80]

high_math.head()

In [ ]:
# 5. loc를 활용한 값 변경: Park의 math_score와 english_score가 반대로 입력.
df.loc[2, "math_score"] = 40
df.loc[2, "english_score"] = 30

df.head()

In [ ]:
# 6. apply 활용: 총점이 150 이상이면 'Pass', 아니면 'Fail'
def check_pass(score):
  if(score >= 150):
    return 'Pass'
  else:
    return "Fail"

df["status"] = df["total_score"].apply(check_pass)
df["status"] = df["total_score"].apply(lambda score: check_pass(score))
df["status"] = df["total_score"].apply(lambda score: "Pass" if score >= 150 else "Fail")

**Pandas 3: 데이터 병합, 그룹화, 상관관계**

In [ ]:
# 7. merge (데이터 병합): 방과 후 활동 데이터와 합치기
df_activity = pd.read_csv('after_school.csv')
merged_df = pd.merge(df, df_activity, on="student_id")
merged_df.head()

In [ ]:
# 8. groupby: 방과 후 활동(activity)별 수학 점수 평균
grouped_math = merged_df.groupby("activity")["math_score"].mean()
grouped_math.head()

In [ ]:
# 9. 상관관계 (corr): 게임 시간과 과목 점수들의 관계 확인 (수치형 데이터만)
# 게임 시간과 점수가 음의 상관관계를 가짐을 확인
numeric_cols = merged_df[["game_hours", "math_score", "english_score"]].corr()
numeric_cols.head()

**필수 통계 개념 (Numpy, Scipy, Statistics)**

In [ ]:
import numpy as np
from scipy import stats
import statistics

# 수학점수의 평균값, 중앙값, 75 백분위, 최빈값, 표준편차를 구하시오.

scores = merged_df["math_score"] # 계산을 위해 필터링

# 1. numpy: 평균, 중앙값, 백분위
print("평균:", np.mean(scores))
print("중앙값:", np.median(scores))
print("75백분위수:", np.percentile(scores, 75))

# 2. scipy.stats: 최빈값
# keepdims=True는 최신 버전 경고 방지용, [0]으로 값만 추출
mode_result = stats.mode(scores, keepdims=True)[0][0]
print("최빈값:", mode_result)

# 3. statistics: 표준편차
stdev_result = statistics.stdev(scores)
print("표준편차:", statistics.stdev(scores))

**데이터 시각화 (Matplotlib & Seaborn)**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 깨짐 방지 및 그래프 크기 설정 (Colab 기본 설정)
plt.rcParams['figure.figsize'] = (10, 8)

# 1. Matplotlib: 원형 그래프 (Pie) - 합격/불합격 비율
plt.subplot(2, 2, 1)
status_counts = merged_df["status"].value_counts()
plt.pie(status_counts, labels=status_counts.index, autopct='%1.1f%%', colors=['#ff9999','#66b3ff'])
plt.title("Pass / Fail Ratio")

# 2.1 Matplotlib: 막대 그래프 (Bar) - 학생별 총점
plt.subplot(2, 2, 2)
plt.bar(merged_df["name"], merged_df["total_score"], color='skyblue')
plt.title("Total Score by Student")

# 2.2 Matplotlib: 막대 그래프 (Bar) - Gender 별 수학 점수 평균점수
#df2 = df.groupby(["gender"])[["math_score", "english_score"]].mean()
#plt.bar(df2.index, df2["math_score"])
#plt.title("Avg Math Score by Gender")

# 3. Seaborn: 산점도 (Scatter) - 게임 시간 vs 수학 점수 (옵션 : 성별, 크기)
plt.subplot(2, 2, 3)
sns.scatterplot(merged_df, x="math_score", y="game_hours", hue="gender", s=100)
plt.title("Game Hours vs Math Score")

# 4. Seaborn: 박스 플롯 (Box) - 성별에 따른 영어 점수 분포 (옵션 : 디자인)
plt.subplot(2, 2, 4)
sns.boxplot(merged_df, x="gender", y="english_score", palette="pastel")
plt.title("English Score by Gender")

plt.tight_layout()
plt.show()

# 5. Seaborn: 히트맵 (Heatmap) - 상관관계 시각화 (별도 출력)
plt.figure(figsize=(6, 4))
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()